In [5]:
# ============================
# COLAB: Coma cluster run (one-shot) with declining T(r)
# Y0_CENTRAL = 0.23  (your requested run)
# ----------------------------
# What it prints:
#   - M_g, M_phi, M_total at r ≈ R500
#   - M_phi/M_g
#   - outer slope d ln rho / d ln r (sanity check)
# ============================

import math
import numpy as np
from scipy.integrate import solve_ivp

# -----------------------------
# Physical constants (SI)
# -----------------------------
G   = 6.674e-11
a0  = 1.2e-10
kB  = 1.380649e-23
m_p = 1.67262192369e-27
mu_gas = 0.6

KPC_M   = 3.085677581e19
MSUN_KG = 1.98847e30

# -----------------------------
# Coma inputs / run knobs
# -----------------------------
T0_keV     = 8.2
R500_kpc   = 1300.0
Y0_CENTRAL = 0.18     # <-- requested value
X0         = 1e-5

RTOL       = 1e-8
ATOL       = 1e-12
MAX_STEP   = 0.2
U_SERIES   = 2e-3

# Numerical guards
LN_Y_FLOOR = -120.0  # prevents exp(-inf)
Y_FLOOR    = 1e-300  # prevents log(0)

# -----------------------------
# Scaling (correct)
#   r0 = (kB*T0)/(mu*m_p*a0) [m]
#   M0 = a0*r0^2/G           [kg]
#   rho0 = a0/(4π G r0)      [kg/m^3]
# -----------------------------
T0_K = T0_keV * 1.16045e7
r0   = (kB * T0_K) / (mu_gas * m_p * a0)
M0   = (a0 * r0 * r0) / G
rho0 = a0 / (4.0 * math.pi * G * r0)

R500_m = R500_kpc * KPC_M
X_MAX  = R500_m / r0

print("=== COMA: DECLINING T(r) + SCALAR SELF-ENERGY ===")
print(f"T0_keV      = {T0_keV:.3f}  (T0_K = {T0_K:.6e} K)")
print(f"r0          = {r0/KPC_M:.3f} kpc")
print(f"R500        = {R500_kpc:.1f} kpc  -> X_MAX = {X_MAX:.6f}")
print(f"rho0        = {rho0:.6e} kg/m^3")
print(f"M0          = {M0/MSUN_KG:.6e} Msun")
print(f"Y0_CENTRAL  = {Y0_CENTRAL:.6f}")
print()

# -----------------------------
# Temperature profile: conservative declining Vikhlinin-like
# theta(x) = T/T0
# Tuned only to be mild (Coma-like): theta(R500) ~ 0.7–0.8
# -----------------------------
x_t = 0.45 * X_MAX
a_T = 0.0
b_T = 2.0
c_T = 0.35

def theta(x: float) -> float:
    if x <= 0.0:
        return 1.0
    z = x / x_t
    return (z ** (-a_T)) * ((1.0 + z**b_T) ** (-c_T / b_T))

def dtheta_dx(x: float) -> float:
    if x <= 0.0:
        return 0.0
    z = x / x_t
    th = theta(x)
    # d/dx ln theta = -(a/x) - (c/b) * d/dx ln(1+z^b)
    term_a = -a_T / x if a_T != 0.0 else 0.0
    dln = term_a - (c_T / b_T) * ( (b_T * z**(b_T-1)) / (x_t * (1.0 + z**b_T)) )
    return th * dln

print(f"theta(R500) = {theta(X_MAX):.6f}  (T(R500) = {theta(X_MAX)*T0_keV:.3f} keV)")
print()

# -----------------------------
# Scalar functions
# mu(Y) = 1 - exp(-Y^(1/4))
# F(Y)  = ∫_0^Y mu(s) ds  (series for small U=Y^(1/4), exact otherwise)
# y_phi(Y) = 0.5*(2Y*mu - F)
# -----------------------------
def U_from_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    return math.exp(0.25 * math.log(Y))

def mu_Y(Y: float) -> float:
    U = U_from_Y(Y)
    return -math.expm1(-U)

def F_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    U = U_from_Y(Y)
    if U < U_SERIES:
        U2 = U*U; U4 = U2*U2
        U5 = U4*U; U6 = U5*U
        U7 = U6*U; U8 = U7*U
        U9 = U8*U; U10 = U9*U
        return 4.0*(U5/5 - U6/12 + U7/42 - U8/192 + U9/1080 - U10/7200)
    e = math.exp(-U)
    poly = (U**3 + 3.0*U**2 + 6.0*U + 6.0)
    return Y - 24.0 + 4.0 * e * poly

def y_phi_from_Y(Y: float) -> float:
    if Y <= 0.0:
        return 0.0
    mu = mu_Y(Y)
    FY = F_Y(Y)
    val = 0.5 * (2.0 * Y * mu - FY)
    if val < 0.0 and val > -1e-14:
        return 0.0
    return max(0.0, val)

# -----------------------------
# ODE system: u = [ln y, mg, mp]
# where:
#   y  = rho_g/rho0
#   mg = M_g/M0
#   mp = M_phi/M0
# -----------------------------
def rhs(x: float, u: np.ndarray) -> np.ndarray:
    ln_y, mg, mp = float(u[0]), float(u[1]), float(u[2])
    if x <= 0.0 or not (math.isfinite(ln_y) and math.isfinite(mg) and math.isfinite(mp)):
        return np.array([0.0, 0.0, 0.0], dtype=np.float64)

    ln_y = max(ln_y, LN_Y_FLOOR)
    y = math.exp(ln_y)

    mg = max(mg, 0.0)
    mp = max(mp, 0.0)
    m_tot = mg + mp

    # ghat = g/a0 using the exact force law:
    # ghat = (m_tot/x^2) / (1 - exp(-sqrt(m_tot)/x))
    s = math.sqrt(max(m_tot, 0.0)) / x
    denom = -math.expm1(-s)
    if denom < 1e-14:
        denom = max(s, 1e-14)

    ghat = (m_tot / (x*x)) / denom
    Y = ghat*ghat
    yphi = y_phi_from_Y(Y)

    th = theta(x)
    dth = dtheta_dx(x)

    dlny_dx = -(ghat/th + dth/th)
    dmg_dx  = x*x*y
    dmp_dx  = x*x*yphi

    return np.array([dlny_dx, dmg_dx, dmp_dx], dtype=np.float64)

# -----------------------------
# Initial conditions near center
# -----------------------------
y0 = float(Y0_CENTRAL)
lny0 = math.log(max(y0, Y_FLOOR))
mg0 = (X0**3) * y0 / 3.0
mp0 = 0.0
u0 = np.array([lny0, mg0, mp0], dtype=np.float64)

# -----------------------------
# Integrate
# -----------------------------
sol = solve_ivp(
    rhs,
    (X0, X_MAX),
    u0,
    method="Radau",
    rtol=RTOL,
    atol=ATOL,
    max_step=MAX_STEP
)

if not sol.success:
    raise RuntimeError("ODE integration failed: " + str(sol.message))

x    = sol.t
ln_y = sol.y[0]
y    = np.exp(np.maximum(ln_y, LN_Y_FLOOR))
mg   = sol.y[1]
mp   = sol.y[2]
m_tot = mg + mp

# -----------------------------
# Report at outer radius (≈ R500)
# -----------------------------
r_final_kpc = (x[-1] * r0) / KPC_M
Mg_Msun     = mg[-1] * (M0 / MSUN_KG)
Mphi_Msun   = mp[-1] * (M0 / MSUN_KG)
Mtot_Msun   = m_tot[-1] * (M0 / MSUN_KG)

print("=== RESULTS at r ≈ R500 ===")
print(f"x_final      = {x[-1]:.6f}")
print(f"r_final      = {r_final_kpc:.3f} kpc")
print(f"M_g          = {Mg_Msun:.3e} Msun")
print(f"M_phi        = {Mphi_Msun:.3e} Msun")
print(f"M_total      = {Mtot_Msun:.3e} Msun")
print(f"M_phi / M_g  = {Mphi_Msun/Mg_Msun:.6f}")
print()

# -----------------------------
# Outer slope check: median of last 10% points
# -----------------------------
slope = np.gradient(np.log(y + 1e-300), np.log(x + 1e-300))
outer_slope = float(np.median(slope[int(0.9*len(x)):]))

print("=== CONSISTENCY CHECKS ===")
print(f"outer slope median d ln rho / d ln r  ≈ {outer_slope:.3f}  (target ~ -2)")
print(f"rho_g(R500)/rho0                      = {y[-1]:.6e}")
print()


=== COMA: DECLINING T(r) + SCALAR SELF-ENERGY ===
T0_keV      = 8.200  (T0_K = 9.515690e+07 K)
r0          = 353.543 kpc
R500        = 1300.0 kpc  -> X_MAX = 3.677059
rho0        = 1.311571e-23 kg/m^3
M0          = 1.076125e+14 Msun
Y0_CENTRAL  = 0.180000

theta(R500) = 0.732166  (T(R500) = 6.004 keV)

=== RESULTS at r ≈ R500 ===
x_final      = 3.677059
r_final      = 1300.000 kpc
M_g          = 1.566e+14 Msun
M_phi        = 1.081e+14 Msun
M_total      = 2.647e+14 Msun
M_phi / M_g  = 0.690232

=== CONSISTENCY CHECKS ===
outer slope median d ln rho / d ln r  ≈ -1.739  (target ~ -2)
rho_g(R500)/rho0                      = 5.179688e-02



In [4]:
import numpy as np
from scipy.integrate import quad

def omega_m_z(z, om0):
    """
    Calculate Omega_m(z) for a flat LCDM background.
    """
    E2 = om0 * (1 + z)**3 + (1 - om0)
    return om0 * (1 + z)**3 / E2

def integrand(z, om0):
    """
    The integrand for the growth suppression factor I(z).
    Integrand = Omega_m(z)^(6/11) * ln(Omega_m(z)) / (1+z)
    """
    omz = omega_m_z(z, om0)
    # Note: ln(Omega_m) is negative for z < infinity
    return (omz**(6/11) * np.log(omz)) / (1 + z)

def calculate_I0(om0):
    """
    Integrate I(0) from z=0 to high redshift.
    We take the absolute value to match the physics requirement
    (I(0) > 0) stated in Document 9 for growth suppression.
    """
    # Integration limit 1000 is sufficient as Omega_m -> 1
    integral_val, error = quad(integrand, 0, 1000, args=(om0,))
    return abs(integral_val)

def run_s8_calculation(om0_central, alpha_lens_values):
    print(f"--- VSU S8 Calculation Run ---")
    print(f"Central Omega_m0: {om0_central}")

    # 1. Calculate the integral I(0)
    I0 = calculate_I0(om0_central)
    print(f"Calculated Integral I(0): {I0:.5f}")

    # 2. Calculate S8 suppression for various alpha values
    print(f"\nResulting S8 Shifts (S8_VSU / S8_GR):")
    print(f"{'alpha_lens':<12} | {'Correction Factor':<18} | {'Percent Change':<15}")
    print("-" * 50)

    for alpha in alpha_lens_values:
        # Formula: 1 - (3/55) * alpha * I0
        correction = 1 - (3/55) * alpha * I0
        percent_change = (correction - 1) * 100
        print(f"{alpha:<12.2f} | {correction:<18.5f} | {percent_change:>13.3f}%")

# --- Execute the Run ---
if __name__ == "__main__":
    # "Yo, central = 0.23 run"
    CENTRAL_OMEGA_M = 0.23

    # Test a range of lensing enhancement parameters (alpha)
    # alpha ~ 0.01 - 0.2 are typical perturbative values
    ALPHAS_TO_TEST = [0.01, 0.05, 0.10, 0.15, 0.23]

    run_s8_calculation(CENTRAL_OMEGA_M, ALPHAS_TO_TEST)

--- VSU S8 Calculation Run ---
Central Omega_m0: 0.23
Calculated Integral I(0): 0.46041

Resulting S8 Shifts (S8_VSU / S8_GR):
alpha_lens   | Correction Factor  | Percent Change 
--------------------------------------------------
0.01         | 0.99975            |        -0.025%
0.05         | 0.99874            |        -0.126%
0.10         | 0.99749            |        -0.251%
0.15         | 0.99623            |        -0.377%
0.23         | 0.99422            |        -0.578%
